# Suite VIS — Visitors protocol conformance

Every class that plays a visitor role must implement its `Visitors` protocol completely, with matching arity:
Proxies builders, Plain writers, the Reachable collector, and the Validators recorder. Proxies instances implement
`Visitable`, not a visitor protocol. TypeScript checks `implements` at compile time; these cases check at runtime.

In [ ]:
import { Plain, Proxies, Reachable, Schemas as S, Validators, Visitors } from "../src/Framework/index.js";
import { NotImplementedError } from "../src/Framework/Errors.js";
import { assert, conformance_problems, instance_problems, PROTOCOLS, raises, same, text } from "./support.js";

## VIS-01 · The protocols declare what Framework.md says

In [ ]:
assert(same(Object.keys(Visitors.PROTOCOL_METHODS).sort(), Object.keys(PROTOCOLS).sort()));
for (const [name, methods] of Object.entries(Visitors.PROTOCOL_METHODS)) {
  assert(same([...methods].sort(), Object.keys(PROTOCOLS[name] as object).sort()), name);
}
assert(same([...Visitors.PROTOCOL_METHODS.OfAny], ["as_native", "as_object", "as_union", "as_intersection"]));
assert(same([...Visitors.PROTOCOL_METHODS.Visitable], ["identity", "schema_name", "accept"]));
assert(!(Visitors.PROTOCOL_METHODS.OfObject as readonly string[]).includes("identity")); // identity belongs to the visited object
{
  // Native is exactly bigint | number | string | boolean | Uint8Array (a compile-time check).
  const natives: Visitors.Native[] = [1n, 1.5, "s", true, new Uint8Array()];
  assert(natives.length === 5);
}

## VIS-02 · Every implementation conforms to its protocol

In [ ]:
const P = Proxies._internals;
const implementations: Record<string, [{ name: string; prototype: object }, string][]> = {
  Proxies: [[P.ObjectBuilderTarget, "OfObject"], [P._AdjacencySlot, "OfAdjacency"], [P._EntryBuilder, "OfEntry"],
    [P._LinkSlot, "OfLink"], [P._PropertySlot, "OfProperty"], [P._AnySlot, "OfAny"], [P._NativeSlot, "OfNative"],
    [P.ObjectTarget, "Visitable"]],
  Plain: [[Plain._ObjectWriter, "OfObject"], [Plain._AdjacencyWriter, "OfAdjacency"], [Plain._EntryWriter, "OfEntry"],
    [Plain._LinkWriter, "OfLink"], [Plain._PropertyWriter, "OfProperty"], [Plain._AnyWriter, "OfAny"], [Plain._NativeWriter, "OfNative"]],
  Reachable: [[Reachable._Collector, "OfObject"], [Reachable._Adjacency, "OfAdjacency"], [Reachable._Entry, "OfEntry"],
    [Reachable._Link, "OfLink"], [Reachable._Ignored, "OfProperty"], [Reachable._Ignored, "OfAny"], [Reachable._Ignored, "OfNative"]],
  Validators: [[Validators._ObjectRecord, "OfObject"], [Validators._AdjacencyRecord, "OfAdjacency"],
    [Validators._EntryRecord, "OfEntry"], [Validators._Link, "OfLink"], [Validators._Value, "OfProperty"],
    [Validators._Value, "OfAny"], [Validators._Value, "OfNative"]],
};
{
  const problems = Object.values(implementations).flat().flatMap(([impl, proto]) => conformance_problems(impl, proto));
  assert(problems.length === 0, problems.join("; "));
}

## VIS-03 · Proxies instances are `Visitable`, not visitors
Framework.md: builders and serializers implement `Visitors`; proxies do not.

In [ ]:
for (const protocol of ["OfObject", "OfAdjacency", "OfEntry"]) {
  assert(conformance_problems(P.ObjectTarget, protocol).length > 0, protocol);
}

## VIS-04 · Visitor methods chain: non-query methods return `this`

In [ ]:
const Rel = new S.OfRelation.Builder().links("a", "b").properties(text("k")).create();
const Obj = new S.OfObject.Builder().properties(text("x")).relations((r) => r.name("r").of(Rel).me("a")).create();
Proxies.register("Obj", Obj);
Proxies.register("Rel", Rel);
{
  const builder = Proxies.Builders.Obj();
  assert(builder.property("x", () => null) === builder);
  assert(builder.properties(() => null) === builder);
  assert(builder.clear("x") === builder);
  assert(builder.adjacencies(() => null) === builder);
  assert(builder.adjacency("r", () => null) === builder);

  const slots: any[] = [];
  builder.property("x", (p: any) => slots.push(p));
  const prop = slots[0];
  assert(prop.value(() => null) === prop && prop.clear() === prop);
  prop.value((a: any) => slots.push(a));
  const any_ = slots[1];
  assert(any_.as_native(() => null) === any_);
  any_.as_native((n: any) => slots.push(n));
  const native = slots[2];
  assert(native.set("v") === native && native.clear() === native);

  builder.adjacency("r", (a: any) => slots.push(a));
  const adjacency = slots[3];
  assert(adjacency.add((e: any) => slots.push(e)) === adjacency);
  const entry = slots[4];
  assert(entry.property("k", () => null) === entry && entry.clear("k") === entry);
  assert(entry.links((link: any) => slots.push(link)) === entry);
  const link = slots[5];
  assert(link.name() === "b" && link.set(Proxies.Builders.Obj().create()) === link);
  assert(adjacency.remove(entry) === adjacency && adjacency.entries(() => null) === adjacency);
}

## VIS-05 · Protocols without implementations yet
`OfUnion`, `OfIntersection` and `OfRelation` are declared but no class implements them; union, intersection and
object-valued properties raise `NotImplementedError` in every implementation (pinned here, see TestPlan.md).

In [ ]:
for (const impl of [P._AnySlot, Plain._AnyWriter, Validators._Value] as { prototype: any }[]) {
  const instance = Object.create(impl.prototype);
  for (const method of ["as_object", "as_union", "as_intersection"]) {
    raises(NotImplementedError, () => impl.prototype[method].call(instance, () => null));
  }
}

## VIS-06 · Live instances conform too
A class can declare every method yet have an instance field shadow one of them (the Python recorder once did:
its `adjacencies` map hid its `adjacencies()` method). This checks method callability on real instances.

In [ ]:
{
  const found: [object, string][] = [];
  const builder = Proxies.Builders.Obj();
  found.push([builder, "OfObject"]);
  builder.property("x", (p: any) => { found.push([p, "OfProperty"]); p.value((a: any) => { found.push([a, "OfAny"]); a.as_native((n: any) => found.push([n, "OfNative"])); }); });
  builder.adjacency("r", (a: any) => { found.push([a, "OfAdjacency"]); a.add((e: any) => { found.push([e, "OfEntry"]); e.link("b", (l: any) => found.push([l, "OfLink"])); }); });
  found.push([Proxies.Builders.Obj().create(), "Visitable"]);
  const record = new Validators._ObjectRecord();
  record.adjacency("r", (a: any) => a.add((e: any) => found.push([e, "OfEntry"])));
  found.push([record, "OfObject"], [new Reachable._Collector(() => null), "OfObject"]);
  const writer = new Plain._ObjectWriter(new Map(), Obj, () => new Map());
  found.push([writer, "OfObject"]);
  writer.adjacency("r", (a: any) => { found.push([a, "OfAdjacency"]); a.add((e: any) => found.push([e, "OfEntry"])); });
  const problems = found.flatMap(([instance, protocol]) => instance_problems(instance, protocol));
  assert(problems.length === 0, problems.join("; "));
  assert(record.adjacencies((a) => a.name()) === record); // the method the Python bug hid
}